# 01 — First look at the West Mercia crime data

This is an **independent** analysis of West Mercia (Herefordshire,
Shropshire, Telford and Wrekin, and Worcestershire), run the same way as
the London analysis in `notebooks/london/` but not assuming any of its
findings carry over. West Mercia is a much smaller, more rural force area
than the Metropolitan Police, so we check everything from scratch rather
than copy-pasting London's conclusions.

In [ ]:
import sys
sys.path.append("../../src")

import pandas as pd
from load_data import load_force_data

wm = load_force_data("west-mercia")
wm.shape

**110,499 rows** — about 9% the size of London's 1.24M. Makes sense: West
Mercia covers roughly 1.3 million people across mostly rural/small-city
counties, versus Greater London's ~9 million in a dense urban area.

In [ ]:
wm.isna().sum()

### Finding 1: same ASB pattern as London

`Crime ID` is missing for 18,112 rows. Checked below — exactly like
London, this is 100% "Anti-social behaviour", the police's deliberate
anonymisation of ASB reports. Not a loading bug.

In [ ]:
wm.loc[wm["Crime ID"].isna(), "Crime type"].value_counts()

### Finding 2: a pattern London did NOT have

2,099 rows are missing `Longitude`, `Latitude`, and `LSOA name` entirely —
London had zero rows like this. Checked what's actually in the `Location`
field for these rows below.

In [ ]:
missing_geo = wm["LSOA name"].isna()
wm.loc[missing_geo, "Location"].value_counts()

In [ ]:
wm.loc[missing_geo, "Crime type"].value_counts()

Every one of these 2,099 rows literally has `Location == "No Location"` —
a distinct data.police.uk value meaning no geography was published for
that crime at all (not even an anonymised point). Unlike the ASB pattern,
this spans every crime type, dominated by "Violence and sexual offences"
(~72%) but not exclusive to it. These rows are still valid for anything
that doesn't need geography (crime type counts, monthly trends), but must
be **excluded from any district-level analysis** — there's nothing to
attribute them to.

In [ ]:
sorted(wm["Month"].unique())

In [ ]:
has_id = wm["Crime ID"].dropna()
has_id.duplicated().sum()

Same pattern as London's ~0.6% duplicate rate — repeated Crime IDs are a
known, real feature of this dataset (one incident filed under multiple
offence categories, or the same crime reappearing across monthly snapshots
with an updated outcome), not double-counted data.

In [ ]:
wm["Crime type"].value_counts()

## Cleaning

Same redundant columns as London (`clean_crime_data` in `src/clean.py` —
reused as-is, since these are generic to the data.police.uk format, not
London-specific).

In [ ]:
from clean import clean_crime_data

wm = clean_crime_data(wm)
wm.columns.tolist()